# Wan 2.2 TI2V-5B Shot Generator (Kaggle 2x T4)

Generates `outputs/shots/<shot_id>_wan.mp4` from `storyboard.json` keyframes using **`Wan-AI/Wan2.2-TI2V-5B-Diffusers`** (`WanImageToVideoPipeline` + `AutoencoderKLWan`), prompted with **`motion_prompt` only** (camera and movement, not character descriptions).

- **Kaggle setup:** Enable **GPU T4 x2** and Internet.
- **Memory safety:** Starts at a low-resolution draft preset and steps down resolution/frames/steps + switches offloading (`balanced` -> `model_offload` -> `sequential_offload`) if CUDA runs out of memory.
- **Checkpointing:** Saves each clip immediately, skips clips that already exist, and logs per-clip timing to `wan_log.jsonl`.
- **Alternative official repo CLI (`Wan-Video/Wan2.2`):** `python generate.py --task ti2v-5B --size 832*480 --ckpt_dir ./Wan2.2-TI2V-5B --image <keyframe> --offload_model True --convert_model_dtype --t5_cpu --prompt "<motion_prompt>"`

In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers.git transformers accelerate sentencepiece protobuf huggingface_hub imageio[ffmpeg]

In [ ]:
from datetime import datetime, timezone
import gc
import json
from pathlib import Path
import time
import torch
from PIL import Image
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video

# Adjust PROJECT_DIR if your repo/dataset is mounted under /kaggle/input/...
PROJECT_DIR = Path("/kaggle/working/short") if Path("/kaggle/working/short").exists() else Path.cwd().parent
STORYBOARD_PATH = PROJECT_DIR / "storyboard.json"
KEYFRAMES_DIR = PROJECT_DIR / "outputs" / "keyframes"
SHOTS_DIR = PROJECT_DIR / "outputs" / "shots"
LOG_PATH = PROJECT_DIR / "wan_log.jsonl"
MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
NEGATIVE_PROMPT = "static, frozen, blurry, distorted, text, watermark, subtitles, low quality"

SHOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project: {PROJECT_DIR} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Draft low-res first, stepping down cleanly if 16GB T4 VRAM runs out.
# Width/height are multiples of 32 (Wan2.2 spatial VAE 16 * patch 2).
PROFILES = [
    {"name": "draft_low", "width": 384, "height": 672, "num_frames": 49, "steps": 20, "mode": "balanced"},
    {"name": "draft_offload", "width": 384, "height": 672, "num_frames": 49, "steps": 20, "mode": "model_offload"},
    {"name": "reduced_seq", "width": 320, "height": 544, "num_frames": 33, "steps": 15, "mode": "seq_offload"},
    {"name": "tiny_seq", "width": 256, "height": 448, "num_frames": 25, "steps": 12, "mode": "seq_offload"},
]

def log_wan(record: dict) -> None:
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def find_keyframe(shot_id: str) -> Path:
    for ext in (".jpg", ".jpeg", ".png"):
        p = KEYFRAMES_DIR / f"{shot_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Missing keyframe for {shot_id} in {KEYFRAMES_DIR}")

def load_pipeline(mode: str):
    # T4 GPUs use native float16 (no native bf16 tensor cores); VAE stays float32 per official HF docs.
    vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
    if mode == "balanced" and torch.cuda.device_count() >= 2:
        pipe = WanImageToVideoPipeline.from_pretrained(
            MODEL_ID, vae=vae, torch_dtype=torch.float16, device_map="balanced"
        )
    else:
        pipe = WanImageToVideoPipeline.from_pretrained(
            MODEL_ID, vae=vae, torch_dtype=torch.float16
        )
        if mode == "seq_offload":
            pipe.enable_sequential_cpu_offload()
        else:
            pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    return pipe

In [ ]:
storyboard = json.loads(STORYBOARD_PATH.read_text(encoding="utf-8"))
fps = int(storyboard.get("format", {}).get("fps", 24))

profile_idx = 0
pipe = None
loaded_mode = None

for shot in storyboard["shots"]:
    sid = shot["id"]
    out_path = SHOTS_DIR / f"{sid}_wan.mp4"
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"{sid}: skipped (already exists: {out_path.name})")
        continue

    kf_path = find_keyframe(sid)
    motion_prompt = shot["motion_prompt"]  # Camera and motion only, not character description
    seed = int(shot["seed"])

    while profile_idx < len(PROFILES):
        prof = PROFILES[profile_idx]
        t0 = time.perf_counter()
        try:
            if pipe is None or loaded_mode != prof["mode"]:
                if pipe is not None:
                    del pipe
                    gc.collect()
                    torch.cuda.empty_cache()
                pipe = load_pipeline(prof["mode"])
                loaded_mode = prof["mode"]

            img = Image.open(kf_path).convert("RGB").resize((prof["width"], prof["height"]))
            gen = torch.Generator(device="cpu").manual_seed(seed)
            frames = pipe(
                image=img,
                prompt=motion_prompt,
                negative_prompt=NEGATIVE_PROMPT,
                height=prof["height"],
                width=prof["width"],
                num_frames=prof["num_frames"],
                num_inference_steps=prof["steps"],
                guidance_scale=5.0,
                generator=gen,
            ).frames[0]

            export_to_video(frames, str(out_path), fps=fps)
            elapsed = round(time.perf_counter() - t0, 3)
            log_wan({
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "shot_id": sid,
                "profile": prof["name"],
                "width": prof["width"],
                "height": prof["height"],
                "num_frames": prof["num_frames"],
                "steps": prof["steps"],
                "seed": seed,
                "latency_seconds": elapsed,
                "output": out_path.name,
                "success": True,
                "error": None,
            })
            print(f"{sid}: saved {out_path.name} ({prof['name']}, {elapsed}s)")
            break
        except RuntimeError as exc:
            elapsed = round(time.perf_counter() - t0, 3)
            is_oom = isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()
            log_wan({
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "shot_id": sid,
                "profile": prof["name"],
                "seed": seed,
                "latency_seconds": elapsed,
                "success": False,
                "error": str(exc),
            })
            if not is_oom or profile_idx + 1 >= len(PROFILES):
                raise
            print(f"{sid}: OOM on {prof['name']}; lowering settings...")
            profile_idx += 1
            if pipe is not None:
                del pipe
                pipe = None
                loaded_mode = None
            gc.collect()
            torch.cuda.empty_cache()